# 01 Data Understanding

This notebook performs an initial profile of the raw dataset before cleaning and modeling.

Scope covered:
- Load data from `data/raw/`
- Dataset size and structure
- Column meaning (business interpretation)
- Data types and missingness
- Range for numeric variables
- Initial observations to flag before preprocessing


In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)


In [6]:
data_dir = Path('../data/raw')
csv_files = sorted(data_dir.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError('No CSV files found in ../data/raw')

# Use the first CSV file in the raw folder for this case study.
dataset_path = csv_files[0]
df = pd.read_csv(dataset_path)

print(f'I have Loaded dataset: {dataset_path.name}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')


I have Loaded dataset: ElectraHub_Data.csv
Shape: 3,000 rows x 16 columns


In [3]:
display(df.head())

,Region,Product_Category,Campaign_Type,Product_Age_Months,Product_Price,Competitor_Price_Index,Advertising_Expenditure,Discount_Percentage,Campaign_Engagement_Score,Inventory_Level,Num_Reviews,Avg_Customer_Rating,Return_Rate,Length_Product_Description,Popularity,Sales
0,South,Tablet,Search Ad,19,1904.04,1.053,425.33,32.95,31.66,1123,370,3.89,2.702,272,Moderate,12265.79
1,West,Tablet,Email,36,1325.95,1.034,600.70,30.97,41.23,480,600,3.99,3.046,311,High,20426.45
2,West,Mobile,Influencer,2,1445.79,1.076,716.22,28.42,51.97,1019,5,4.04,2.856,212,High,29827.18
3,East,Tablet,Search Ad,9,831.62,1.112,705.51,24.39,64.86,692,168,3.99,3.124,202,Moderate,28211.95
4,North,Mobile,Email,30,1582.20,0.850,753.36,27.00,57.00,722,832,3.89,3.215,374,High,29794.80


## Structured Summary

In [4]:
column_meanings = {
    'Region': 'Geographic sales region for the product campaign.',
    'Product_Category': 'Type of product being sold (for example, Mobile, Tablet).',
    'Campaign_Type': 'Marketing channel/campaign format used.',
    'Product_Age_Months': 'Product age in months since launch/listing.',
    'Product_Price': 'Listed selling price of the product.',
    'Competitor_Price_Index': 'Relative competitor pricing index around 1.0 (higher means competitors priced higher).',
    'Advertising_Expenditure': 'Marketing spend allocated for promotion.',
    'Discount_Percentage': 'Discount offered as a percent of listed price.',
    'Campaign_Engagement_Score': 'Engagement score captured for campaign response.',
    'Inventory_Level': 'Units available/in stock level.',
    'Num_Reviews': 'Count of customer reviews received.',
    'Avg_Customer_Rating': 'Average customer rating score.',
    'Return_Rate': 'Product return rate percentage.',
    'Length_Product_Description': 'Length of product description text (likely character count).',
    'Popularity': 'Categorical popularity label (for example Low/Moderate/High).',
    'Sales': 'Target variable: sales outcome amount.'
}

summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [str(df[c].dtype) for c in df.columns],
    'non_null_count': [int(df[c].notna().sum()) for c in df.columns],
    'missing_count': [int(df[c].isna().sum()) for c in df.columns],
    'unique_values': [int(df[c].nunique(dropna=True)) for c in df.columns],
    'column_represents': [column_meanings.get(c, 'Business meaning to be confirmed with domain owner.') for c in df.columns]
})

summary

,column,dtype,non_null_count,missing_count,unique_values,column_represents
0,Region,object,3000,0,4,Geographic sales region for the product campaign.
1,Product_Category,object,3000,0,2,"Type of product being sold (for example, Mobil..."
2,Campaign_Type,object,3000,0,4,Marketing channel/campaign format used.
3,Product_Age_Months,int64,3000,0,36,Product age in months since launch/listing.
4,Product_Price,float64,3000,0,2874,Listed selling price of the product.
5,Competitor_Price_Index,float64,3000,0,399,Relative competitor pricing index around 1.0 (...
6,Advertising_Expenditure,float64,3000,0,2832,Marketing spend allocated for promotion.
7,Discount_Percentage,float64,3000,0,1550,Discount offered as a percent of listed price.
8,Campaign_Engagement_Score,float64,3000,0,2308,Engagement score captured for campaign response.
9,Inventory_Level,int64,3000,0,1092,Units available/in stock level.


In [5]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

numeric_range = pd.DataFrame({
    'column': num_cols,
    'min': [df[c].min() for c in num_cols],
    'max': [df[c].max() for c in num_cols],
    'mean': [df[c].mean() for c in num_cols],
    'median': [df[c].median() for c in num_cols],
    'std': [df[c].std() for c in num_cols]
})

numeric_range.sort_values('column').reset_index(drop=True)

,column,min,max,mean,median,std
0,Advertising_Expenditure,325.340,820.000,623.781673,623.8000,84.686263
1,Avg_Customer_Rating,3.570,4.420,3.984093,3.9800,0.132117
2,Campaign_Engagement_Score,5.720,99.000,52.344707,52.1900,14.614951
3,Competitor_Price_Index,0.780,1.250,0.999106,0.9980,0.080977
4,Discount_Percentage,9.250,42.000,29.439540,29.5350,5.260201
5,Inventory_Level,50.000,1840.000,821.361333,822.0000,290.326387
6,Length_Product_Description,40.000,448.000,247.311000,248.0000,63.098585
7,Num_Reviews,5.000,2989.000,418.200000,346.5000,300.456336
8,Product_Age_Months,1.000,36.000,11.316000,9.0000,8.289695
9,Product_Price,10.110,2000.000,980.839717,906.9350,571.794980


In [7]:
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

cat_snapshot = pd.DataFrame({
    'column': cat_cols,
    'unique_values': [df[c].nunique(dropna=True) for c in cat_cols],
    'top_value': [df[c].mode(dropna=True).iloc[0] if not df[c].mode(dropna=True).empty else np.nan for c in cat_cols],
    'top_value_freq': [df[c].value_counts(dropna=True).iloc[0] if not df[c].value_counts(dropna=True).empty else np.nan for c in cat_cols]
})

cat_snapshot

,column,unique_values,top_value,top_value_freq
0,Region,4,West,845
1,Product_Category,2,Mobile,1884
2,Campaign_Type,4,Social Media,937
3,Popularity,5,High,990


## Initial Observations To Flag Before Cleaning

In [9]:
observations = []

rows, cols = df.shape
observations.append(f'- Dataset blah has **{rows:,} rows** and **{cols} columns**.')

missing_total = int(df.isna().sum().sum())
if missing_total == 0:
    observations.append('- No missing values detected in the raw data.')
else:
    observations.append(f'- Missing values detected: **{missing_total:,}** total. Plan imputation rules by column type.')

dup_count = int(df.duplicated().sum())
if dup_count == 0:
    observations.append('- No fully duplicated rows found.')
else:
    observations.append(f'- Found **{dup_count:,}** exact duplicate rows. Review before model training.')

# Outlier signal using IQR fences
outlier_signals = {}
for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = int(((df[c] < lower) | (df[c] > upper)).sum())
    if outliers > 0:
        outlier_signals[c] = outliers

if outlier_signals:
    top_outliers = sorted(outlier_signals.items(), key=lambda x: x[1], reverse=True)[:4]
    formatted = ', '.join([f"{k} ({v})" for k, v in top_outliers])
    observations.append(f'- Potential outliers exist (IQR rule), strongest in: {formatted}.')
else:
    observations.append('- No strong outlier signals from IQR checks on numeric columns.')

# Target relation quick signal
if 'Sales' in num_cols:
    corr = df[num_cols].corr(numeric_only=True)['Sales'].drop('Sales').sort_values(key=np.abs, ascending=False)
    top_corr = corr.head(5)
    corr_text = ', '.join([f"{idx} ({val:+.2f})" for idx, val in top_corr.items()])
    observations.append(f'- Strongest linear signals vs `Sales` (absolute correlation): {corr_text}.')

# Categorical readiness
if cat_cols:
    high_card = [c for c in cat_cols if df[c].nunique(dropna=True) > 20]
    if high_card:
        observations.append(f'- High-cardinality categorical columns detected: {", ".join(high_card)} (may need careful encoding).')
    else:
        observations.append('- Categorical columns appear low-cardinality and suitable for one-hot encoding.')

print('\n'.join(observations))

- Dataset blah has **3,000 rows** and **16 columns**.
- No missing values detected in the raw data.
- No fully duplicated rows found.
- Potential outliers exist (IQR rule), strongest in: Num_Reviews (119), Product_Age_Months (113), Return_Rate (22), Length_Product_Description (21).
- Strongest linear signals vs `Sales` (absolute correlation): Campaign_Engagement_Score (+0.72), Advertising_Expenditure (+0.65), Avg_Customer_Rating (+0.44), Discount_Percentage (+0.27), Return_Rate (-0.16).
- Categorical columns appear low-cardinality and suitable for one-hot encoding.


In [12]:
# Keep one convenient describe table for quick reference.
display(df.describe(include='all').transpose())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Region,3000,4,West,845,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Category,3000,2,Mobile,1884,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Campaign_Type,3000,4,Social Media,937,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Product_Age_Months,3000.0,NaN,NaN,NaN,11.316,8.289695,1.0,5.0,9.0,15.0,36.0
Product_Price,3000.0,NaN,NaN,NaN,980.839717,571.79498,10.11,526.575,906.935,1475.72,2000.0
Competitor_Price_Index,3000.0,NaN,NaN,NaN,0.999106,0.080977,0.78,0.944,0.998,1.055,1.25
Advertising_Expenditure,3000.0,NaN,NaN,NaN,623.781673,84.686263,325.34,564.3575,623.8,683.4,820.0
Discount_Percentage,3000.0,NaN,NaN,NaN,29.43954,5.260201,9.25,25.83,29.535,33.07,42.0
Campaign_Engagement_Score,3000.0,NaN,NaN,NaN,52.344707,14.614951,5.72,42.26,52.19,62.57,99.0
Inventory_Level,3000.0,NaN,NaN,NaN,821.361333,290.326387,50.0,627.0,822.0,1020.25,1840.0
